# ASRA v0.2 — Phase 1 ARC-AGI-3 (Kaggle-ready)

**Competition:** [ARC Prize 2026 — ARC-AGI-3](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3)

Transition-centric adaptive reasoning for ARC-AGI-3:

- `(state, action, next_state)` transition logs
- state graph world model
- **state-conditioned** latent action semantics inference
- uncertainty-aware exploration + dead-end avoidance
- official **ARC-AGI-3-Agents** / `arc_agi` integration on Kaggle

> Phase 1 research baseline — not a fully optimized solver.

**Video:** [Action semantics inference](https://youtu.be/VmQygZPgK5A)

### Upload steps

1. Competition → **Code** → **New Notebook**
2. **Add data** → `arc-prize-2026-arc-agi-3`
3. Upload this notebook (or paste cells)
4. **Save & Run All** → **Submit to Competition**

## 0. Kaggle bootstrap

In [ ]:
import os
import sys
import glob
import subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/input")
COMP_SLUG = "arc-prize-2026-arc-agi-3"

if IS_KAGGLE:
    COMP_ROOT = f"/kaggle/input/{COMP_SLUG}"
else:
    _candidates = [
        Path(os.environ.get("ASRA_COMP_ROOT", "")),
        Path.cwd() / "private/kaggle-dataset/competition",
        Path.cwd() / "private/kaggle-dataset" / "competition",
    ]
    COMP_ROOT = str(next((p.resolve() for p in _candidates if str(p) and p.is_dir()), Path("private/kaggle-dataset/competition")))

AGENTS_ROOT = os.path.join(COMP_ROOT, "ARC-AGI-3-Agents")
WHEELS_DIR = os.path.join(COMP_ROOT, "arc_agi_3_wheels")
ENV_DIR = os.path.join(COMP_ROOT, "environment_files")
WORKING = "/kaggle/working" if IS_KAGGLE else "."
RECORDINGS_DIR = os.path.join(WORKING, "recordings")
os.makedirs(RECORDINGS_DIR, exist_ok=True)

print("IS_KAGGLE:", IS_KAGGLE)
print("COMP_ROOT:", COMP_ROOT, "| exists:", os.path.isdir(COMP_ROOT))

if os.path.isdir(WHEELS_DIR):
    wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *wheels])
        print("Installed wheels:", len(wheels))
else:
    print("No wheels dir — mock fallback unless arc_agi already installed")

if os.path.isdir(AGENTS_ROOT) and AGENTS_ROOT not in sys.path:
    sys.path.insert(0, AGENTS_ROOT)

os.environ.setdefault("RECORDINGS_DIR", RECORDINGS_DIR)
os.environ.setdefault("ENVIRONMENTS_DIR", ENV_DIR if os.path.isdir(ENV_DIR) else "environment_files")
os.environ["OPERATION_MODE"] = "COMPETITION" if IS_KAGGLE else os.environ.get("OPERATION_MODE", "OFFLINE")
print("OPERATION_MODE:", os.environ["OPERATION_MODE"])

## 1. Configuration

In [ ]:
import json
import time
import random
import hashlib
from dataclasses import dataclass, asdict, field
from collections import defaultdict, Counter
from typing import Any, Dict, List, Tuple, Optional

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

USE_MOCK_ENV = os.environ.get("ASRA_FORCE_MOCK", "0") == "1"
if not USE_MOCK_ENV and not os.path.isdir(COMP_ROOT):
    USE_MOCK_ENV = True
    print("Auto-enabled mock: competition bundle not found at", COMP_ROOT)
MAX_ACTIONS_PER_GAME = int(os.environ.get("ASRA_MAX_ACTIONS", "80"))
MAX_GAMES = int(os.environ.get("ASRA_MAX_GAMES", "3" if not IS_KAGGLE else "9999"))
GAME_FILTER = os.environ.get("ASRA_GAME_FILTER", "")

OUTPUT_DIR = WORKING
TRANSITION_LOG_PATH = os.path.join(OUTPUT_DIR, "asra_v0_2_transition_log.jsonl")
STATE_GRAPH_PATH = os.path.join(OUTPUT_DIR, "asra_v0_2_state_graph.json")
SUMMARY_PATH = os.path.join(OUTPUT_DIR, "asra_v0_2_summary.json")
SCORECARD_PATH = os.path.join(OUTPUT_DIR, "asra_v0_2_scorecard.json")
SUBMISSION_META_PATH = os.path.join(OUTPUT_DIR, "submission_meta.json")
SIMPLE_ACTIONS = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]

print("USE_MOCK_ENV:", USE_MOCK_ENV, "| MAX_GAMES:", MAX_GAMES)

## 2. State utilities

In [ ]:
def canonical_grid(grid: Any) -> List[List[int]]:
    arr = np.array(grid, dtype=int)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D grid, got {arr.shape}")
    return arr.tolist()

def state_hash(grid: Any) -> str:
    payload = json.dumps(canonical_grid(grid), separators=(",", ":"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def grid_diff(before: Any, after: Any) -> Dict[str, Any]:
    b, a = np.array(before, dtype=int), np.array(after, dtype=int)
    if b.shape != a.shape:
        return {"shape_changed": True, "num_changed_cells": None, "changed_cells": [], "change_ratio": None}
    changed = [{"y": int(y), "x": int(x), "from": int(b[y, x]), "to": int(a[y, x])} for y, x in zip(*np.where(b != a))]
    return {"shape_changed": False, "num_changed_cells": len(changed), "change_ratio": float(len(changed) / b.size) if b.size else 0.0, "changed_cells": changed}

## 3. Transition memory

In [ ]:
@dataclass
class TransitionRecord:
    episode_id: str
    step_id: int
    action: str
    state_hash: str
    next_state_hash: str
    state: List[List[int]]
    next_state: List[List[int]]
    diff: Dict[str, Any]
    reward: float = 0.0
    done: bool = False
    info: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)

class TransitionStore:
    def __init__(self):
        self.records: List[TransitionRecord] = []
    def add(self, r: TransitionRecord):
        self.records.append(r)
    def save_jsonl(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            for r in self.records:
                f.write(json.dumps(asdict(r), separators=(",", ":")) + "\n")

class StateGraph:
    def __init__(self):
        self.nodes: Dict[str, Dict[str, Any]] = {}
        self.edges: Dict[Tuple[str, str, str], Dict[str, Any]] = {}
    def add_transition(self, r: TransitionRecord):
        self.nodes.setdefault(r.state_hash, {"visits": 0})
        self.nodes[r.state_hash]["visits"] += 1
        key = (r.state_hash, r.action, r.next_state_hash)
        e = self.edges.setdefault(key, {"count": 0, "rewards": []})
        e["count"] += 1
        e["rewards"].append(float(r.reward))
    def export(self):
        return {"num_nodes": len(self.nodes), "num_edges": len(self.edges)}
    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.export(), f, indent=2)

## 4. Semantics + exploration (v0.2)

In [ ]:
class ActionSemanticsInferencer:
    def __init__(self):
        self.effects: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)
    def observe(self, record: TransitionRecord):
        self.effects[(record.state_hash, record.action)].append({
            "num_changed_cells": record.diff.get("num_changed_cells"),
            "reward": record.reward,
        })
    def infer(self, state_hash_value: str, action: str) -> Dict[str, Any]:
        effects = self.effects.get((state_hash_value, action), [])
        if not effects:
            return {"observations": 0, "hypothesis": "unknown", "consistency_score": None}
        counts = [e["num_changed_cells"] for e in effects if e["num_changed_cells"] is not None]
        std = float(np.std(counts)) if counts else 0.0
        mean = float(np.mean(counts)) if counts else None
        if mean == 0:
            hyp = "no-op / blocked"
        elif mean is not None and mean <= 1.5:
            hyp = "localized cell update"
        else:
            hyp = "multi-cell transform"
        return {"observations": len(effects), "hypothesis": hyp, "consistency_score": float(1.0 / (1.0 + std)) if counts else None}

class ASRAExplorer:
    def __init__(self, action_names: List[str]):
        self.action_names = action_names
        self.state_action_counts = Counter()
        self.action_rewards = defaultdict(list)
        self.dead_ends = set()
    def update(self, record: TransitionRecord):
        self.state_action_counts[(record.state_hash, record.action)] += 1
        self.action_rewards[record.action].append(float(record.reward))
        if record.diff.get("num_changed_cells") == 0 and record.reward <= 0:
            self.dead_ends.add((record.state_hash, record.action))
    def choose_action(self, state_hash_value: str, semantics: ActionSemanticsInferencer, available: Optional[List[str]] = None) -> str:
        candidates = [a for a in self.action_names if available is None or a in available] or list(self.action_names)
        scores = {}
        for action in candidates:
            if (state_hash_value, action) in self.dead_ends:
                continue
            sem = semantics.infer(state_hash_value, action)
            c = sem.get("consistency_score")
            uncertainty = 1.0 if c is None else (1.0 - min(1.0, c))
            local = self.state_action_counts[(state_hash_value, action)]
            mean_r = float(np.mean(self.action_rewards[action])) if self.action_rewards[action] else 0.0
            scores[action] = 2.0 / (1.0 + local) + 0.7 * uncertainty + 0.5 * mean_r + random.random() * 0.05
        return max(scores.items(), key=lambda kv: kv[1])[0] if scores else random.choice(candidates)

## 5. Mock fallback

In [ ]:
class MockARCAGI3Env:
    def __init__(self, max_steps=80):
        self.max_steps = max_steps
        self.step_count = 0
        self.grid = None
    def reset(self):
        self.step_count = 0
        self.grid = np.zeros((3, 3), dtype=int)
        self.grid[1, 1] = 1
        return self.grid.tolist()
    def step(self, action: str):
        self.step_count += 1
        g = self.grid.copy()
        if action == "ACTION1": g[0, 0] = (g[0, 0] + 1) % 4
        elif action == "ACTION2": g[0, 1] = (g[0, 1] + 1) % 4
        elif action == "ACTION3": g[1, 1] = (g[1, 1] + 1) % 4
        elif action == "ACTION4" and g[1, 1]: g[2, 1], g[1, 1] = g[1, 1], 0
        elif action == "ACTION5":
            nz = list(zip(*np.where(g != 0)))
            if nz:
                y, x = random.choice(nz); g[y, x] = 0
        self.grid = g
        done = bool(g[0, 0] and g[0, 1] and g[1, 1]) or self.step_count >= self.max_steps
        return self.grid.tolist(), float(done), done, {}

## 6. Official ASRAAgent

In [ ]:
HAS_OFFICIAL = False
try:
    from arcengine import FrameData, GameAction, GameState
    from agents.agent import Agent
    from agents.swarm import Swarm
    from agents import AVAILABLE_AGENTS
    from arc_agi import Arcade
    HAS_OFFICIAL = True
except Exception as exc:
    print("Official toolkit unavailable:", exc)

GLOBAL_STORE = TransitionStore()
GLOBAL_GRAPH = StateGraph()
GLOBAL_SEMANTICS = ActionSemanticsInferencer()
GLOBAL_EXPLORER = ASRAExplorer(SIMPLE_ACTIONS)

if HAS_OFFICIAL:
    class ASRAAgent(Agent):
        MAX_ACTIONS = MAX_ACTIONS_PER_GAME

        def is_done(self, frames, latest_frame):
            return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS

        def _available_simple(self, latest_frame):
            avail = getattr(latest_frame, "available_actions", None) or []
            names = [a.name for a in avail if hasattr(a, "name") and a.name in SIMPLE_ACTIONS]
            return names or SIMPLE_ACTIONS

        def _to_game_action(self, action_name, grid):
            ga = getattr(GameAction, action_name)
            if ga.is_complex():
                h, w = len(grid), len(grid[0]) if grid else 0
                ga.set_data({"x": w // 2, "y": h // 2})
            ga.reasoning = f"ASRA v0.2: {action_name}"
            return ga

        def choose_action(self, frames, latest_frame):
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                return GameAction.RESET
            grid = latest_frame.frame
            name = GLOBAL_EXPLORER.choose_action(state_hash(grid), GLOBAL_SEMANTICS, self._available_simple(latest_frame))
            return self._to_game_action(name, grid)

        def take_action(self, action):
            self._last_action_name = action.name
            return super().take_action(action)

        def append_frame(self, frame):
            super().append_frame(frame)
            if len(self.frames) < 2:
                return
            prev, curr = self.frames[-2], self.frames[-1]
            if not prev.frame or not curr.frame:
                return
            record = TransitionRecord(
                episode_id=self.game_id,
                step_id=self.action_counter,
                action=getattr(self, "_last_action_name", "UNKNOWN"),
                state_hash=state_hash(prev.frame),
                next_state_hash=state_hash(curr.frame),
                state=canonical_grid(prev.frame),
                next_state=canonical_grid(curr.frame),
                diff=grid_diff(prev.frame, curr.frame),
                reward=float(getattr(curr, "levels_completed", 0) or 0),
                done=curr.state is GameState.WIN,
            )
            GLOBAL_STORE.add(record)
            GLOBAL_GRAPH.add_transition(record)
            GLOBAL_SEMANTICS.observe(record)
            GLOBAL_EXPLORER.update(record)

    AVAILABLE_AGENTS["asra"] = ASRAAgent
    print("Registered ASRAAgent")

## 7. Run

In [ ]:
scorecard = None
run_mode = "mock"

if USE_MOCK_ENV or not HAS_OFFICIAL or not os.path.isdir(COMP_ROOT):
    print("MOCK validation run")
    env = MockARCAGI3Env(max_steps=MAX_ACTIONS_PER_GAME)
    state = canonical_grid(env.reset())
    for step in range(MAX_ACTIONS_PER_GAME):
        action = GLOBAL_EXPLORER.choose_action(state_hash(state), GLOBAL_SEMANTICS)
        nxt, reward, done, _ = env.step(action)
        record = TransitionRecord("mock", step, action, state_hash(state), state_hash(nxt), state, canonical_grid(nxt), grid_diff(state, nxt), float(reward), bool(done))
        GLOBAL_STORE.add(record); GLOBAL_GRAPH.add_transition(record); GLOBAL_SEMANTICS.observe(record); GLOBAL_EXPLORER.update(record)
        state = canonical_grid(nxt)
        if done: break
else:
    print("OFFICIAL Swarm run")
    run_mode = "official"
    arcade = Arcade()
    games = arcade.get_environments()
    if GAME_FILTER:
        games = [g for g in games if any(g.startswith(p.strip()) for p in GAME_FILTER.split(",") if p.strip())]
    games = games[:MAX_GAMES]
    print("Games:", games[:10], f"(total {len(games)})")
    swarm = Swarm("asra", os.environ.get("ARC_BASE_URL", "https://three.arcprize.org"), games, tags=["asra-v0.2"])
    scorecard = swarm.main()

print("mode:", run_mode, "| transitions:", len(GLOBAL_STORE.records))

## 8. Outputs

In [ ]:
GLOBAL_STORE.save_jsonl(TRANSITION_LOG_PATH)
GLOBAL_GRAPH.save(STATE_GRAPH_PATH)
summary = {"agent": "ASRA v0.2", "run_mode": run_mode, "transitions": len(GLOBAL_STORE.records), "graph": GLOBAL_GRAPH.export()}
if scorecard is not None:
    payload = scorecard.model_dump() if hasattr(scorecard, "model_dump") else scorecard
    with open(SCORECARD_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    summary["scorecard"] = True
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
meta = {"competition": COMP_SLUG, "agent": "asra", "version": "v0.2", "run_mode": run_mode}
with open(SUBMISSION_META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(summary, indent=2))

## 9. Next steps

Object-centric diffs, goal inference, graph planner, per-game semantics. Same framework extends to Decision Biology / NFM.